# Sesión 2 — Soluciones: RAG y Diseño de Caso de Uso

Versión con soluciones completas para el instructor.

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "gemini"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3:4b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash + gemini-embedding-001")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    llm = ChatOllama(model=OLLAMA_MODEL)
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL} + {OLLAMA_EMBEDDING_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}")

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Cargando 200 reviews de Amazon Electronics...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(200))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

print(f"✓ {len(df)} reviews listas")

## Solución Ejercicio 1: Crear los documentos

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={
            "rating": row["rating"],
            "parent_asin": row["parent_asin"]
        }
    )
    for _, row in df.iterrows()
]

print(f"Documentos creados: {len(docs)}")
print(f"\nPrimer documento:")
print(f"  Content: {docs[0].page_content[:200]}")
print(f"  Metadata: {docs[0].metadata}")

## Solución Ejercicio 2: Construir el vector store

In [ ]:
from langchain_chroma import Chroma

print("Indexando documentos...")
vectorstore = Chroma.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✓ Vector store con {vectorstore._collection.count()} vectores")

test_docs = retriever.invoke("auriculares bluetooth")
print(f"✓ Retriever devuelve {len(test_docs)} documentos para 'auriculares bluetooth'")

## Solución Ejercicio 3: Construir la cadena RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:400]}"
        for d in docs
    )

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto en análisis de productos de electrónica.
Responde usando SOLO la información de las siguientes reviews de clientes reales.
Si la información no aparece en las reviews, dilo explícitamente.

Reviews recuperadas:
{context}"""),
    ("human", "{question}")
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✓ Cadena RAG construida")

## Solución Ejercicio 4: Prueba y comparación

In [ ]:
mi_pregunta = "¿Qué problemas tienen los altavoces portátiles según los clientes?"

print("=== SIN RAG ===")
print(llm.invoke(mi_pregunta).content)

print("\n" + "=" * 60)
print("=== CON RAG ===")
print(rag_chain.invoke(mi_pregunta))

## Solución Ejercicio 5 (bonus): Filtrado por metadata

In [ ]:
# ChromaDB usa operadores de comparación dentro de un dict "where"
retriever_negativo = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "filter": {"rating": {"$lte": 2}}
    }
)

rag_chain_negativo = (
    {"context": retriever_negativo | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

pregunta_negativa = "¿Qué falló en estos productos?"

print("=== RAG (solo reviews negativas, rating ≤ 2) ===")
print(rag_chain_negativo.invoke(pregunta_negativa))

---

## Parte 2: Ejemplo de caso de uso completado

*(Ejemplo de referencia para el instructor — el caso es ficticio pero ilustra el nivel de detalle esperado)*

### Caso de uso

**Nombre del caso de uso:**  
Asistente de primera respuesta a quejas de clientes en e-commerce

**Función de negocio:**  
Atención al cliente

**Descripción del problema actual:**  
El equipo de atención al cliente recibe ~500 tickets/día. El 70% son consultas repetidas (estado de pedido, política de devoluciones, compatibilidad de productos). Un agente tarda 8 minutos de media por ticket; el backlog crece los lunes y después de campañas. La insatisfacción es alta por los tiempos de espera.

**Propuesta de solución:**  
Sistema RAG que indexa el catálogo de productos, la política de devoluciones y el historial de tickets resueltos. Para cada nuevo ticket, recupera el contexto relevante y genera un borrador de respuesta. El agente humano revisa y envía (human-in-the-loop). Para las consultas más frecuentes y de bajo riesgo, el sistema responde automáticamente y el agente solo ve los casos escalados.

### Arquitectura propuesta

**Estrategia principal:**  
RAG (el conocimiento del catálogo y política de devoluciones cambia frecuentemente; fine-tuning sería demasiado lento y caro de mantener)

**Fuentes de datos:**  
- Catálogo de productos (exportación del ERP, actualización diaria)
- Documento de política de devoluciones y envíos (actualización mensual)
- Historial de los últimos 1.000 tickets resueltos con su respuesta correcta

**Modelo y proveedor:**  
Gemini 2.5 Flash para borradores de respuesta (velocidad y coste bajo). Gemini Embedding 001 para indexado. Evaluaremos migrar a Ollama si el volumen de tokens hace el coste inasumible.

**¿Build o Buy?**  
Build con LangChain + ChromaDB. Los datos de clientes son sensibles (RGPD); no podemos usar una solución SaaS sin un DPA sólido. El volumen justifica la inversión en infraestructura propia a partir del mes 3.

### Estimación de impacto

**Impacto esperado en tiempo / coste:**  
De 8 minutos a ~2 minutos por ticket para los casos que requieren revisión humana. Los tickets automáticos (estimamos 40% del volumen) se resuelven en <30 segundos. Ahorro estimado: 3 FTE equivalentes.

**Métricas de éxito:**  
- Tiempo medio de resolución (objetivo: <3 min)
- CSAT (Customer Satisfaction Score) — mantener igual o mejorar
- Tasa de escalado a humano (referencia inicial: 60%, objetivo: 40%)
- Faithfulness de las respuestas RAG (evaluación mensual con muestra aleatoria)

**¿Necesita supervisión humana?**  
Sí. Fase 1: todos los tickets pasan por revisión humana antes de enviarse. Fase 2: solo los tickets de alto valor o fuera del dominio conocido se escalan. Los tickets de reclamaciones económicas (devoluciones >150€) siempre pasan por humano.

### Roadmap a 3 meses

| Mes | Hito |
|-----|------|
| Mes 1 | Pipeline RAG en staging. Evaluación de faithfulness con 100 tickets históricos. DPA firmado con Google. |
| Mes 2 | Piloto con 20% del volumen real. Agentes humanos revisan todos los borradores. Métricas de baseline. |
| Mes 3 | Automatización del 40% de menor riesgo. Dashboard de métricas en producción. Revisión de costes y decisión de escalar. |

### Riesgos y mitigaciones

| Riesgo | Probabilidad | Impacto | Mitigación |
|--------|-------------|---------|------------|
| El modelo alucina información de devoluciones y un cliente actúa en base a ello | Media | Alto | Revisión humana obligatoria en Fase 1; prompt estricto de no inventar información |
| El catálogo no se actualiza a tiempo y el RAG responde con precios o stocks obsoletos | Alta | Medio | Pipeline de ingestión automatizado diario; timestamp visible en las respuestas |
| Coste de API superior al esperado en picos de demanda (Black Friday) | Media | Medio | Rate limiting, caching de respuestas frecuentes, alertas de gasto |

**¿Hay datos de clientes o empleados involucrados?**  
Sí. Los tickets contienen nombre, email y datos de pedido (datos personales bajo RGPD). Necesitamos: (1) DPA con Google antes del piloto, (2) anonimizar tickets históricos antes de indexarlos, (3) revisar ToS para confirmar que Google no usa estos datos para entrenamiento en el tier empresarial.